## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 13: Hyperparameter-Optimierung
Niveau: Fortgeschrittene
Aufgabe 13.1: Bayesianische Optimierung manuell

Lernziel: Bayesianische Optimierung mit scipy verstehen
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from scipy.stats import norm
from scipy.optimize import minimize

(X_train, y_train), _ = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_sub = X_train[:3000]
y_sub = y_train[:3000]

def evaluiere(log_lr):
    """Testet eine Lernrate und gibt Val-Accuracy zurück."""
    lr = 10 ** log_lr
    model = keras.Sequential([
        keras.layers.Dense(64, activation='relu', input_shape=(784,)),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    h = model.fit(X_sub, y_sub, epochs=10, batch_size=64,
                  validation_split=0.2, verbose=0)
    return max(h.history['val_accuracy'])

# Manuelle Bayesianische Optimierung (Gaussian Process Surrogate)
class GaussianProcessSurrogate:
    def __init__(self, noise=1e-5):
        self.X_obs = []
        self.y_obs = []
        self.noise = noise
    
    def rbf_kernel(self, X1, X2, length_scale=0.5, sigma=1.0):
        dists = np.sum((X1[:, None] - X2[None, :])**2, axis=-1)
        return sigma**2 * np.exp(-0.5 * dists / length_scale**2)
    
    def fit(self, X, y):
        self.X_obs = np.array(X).reshape(-1, 1)
        self.y_obs = np.array(y)
    
    def predict(self, X_new):
        X_new = np.array(X_new).reshape(-1, 1)
        if len(self.X_obs) == 0:
            return np.zeros(len(X_new)), np.ones(len(X_new))
        
        K_obs = self.rbf_kernel(self.X_obs, self.X_obs) + self.noise * np.eye(len(self.X_obs))
        K_star = self.rbf_kernel(X_new, self.X_obs)
        K_ss   = self.rbf_kernel(X_new, X_new)
        
        K_inv = np.linalg.inv(K_obs)
        mu    = K_star @ K_inv @ self.y_obs
        var   = np.diag(K_ss - K_star @ K_inv @ K_star.T)
        return mu, np.maximum(var, 1e-10)
    
    def upper_confidence_bound(self, X_new, kappa=2.576):
        mu, var = self.predict(X_new)
        return mu + kappa * np.sqrt(var)

# Bayesianische Optimierung
gp = GaussianProcessSurrogate()
X_beobachtet = []
y_beobachtet = []
suchraum = np.linspace(-4, -1, 100)

# Initial: 3 zufällige Punkte
print("=== Bayesianische Optimierung der Lernrate ===")
for _ in range(3):
    x_neu = np.random.uniform(-4, -1)
    y_neu = evaluiere(x_neu)
    X_beobachtet.append(x_neu)
    y_beobachtet.append(y_neu)
    print(f"  LR=10^{x_neu:.2f}: Val Acc={y_neu:.4f}")

# 7 BO-Schritte
for schritt in range(7):
    gp.fit(X_beobachtet, y_beobachtet)
    ucb = gp.upper_confidence_bound(suchraum.reshape(-1, 1))
    
    # Nächster Punkt: Maximum des UCB
    x_naechst = suchraum[np.argmax(ucb)]
    y_naechst = evaluiere(x_naechst)
    X_beobachtet.append(x_naechst)
    y_beobachtet.append(y_naechst)
    print(f"Schritt {schritt+1}: LR=10^{x_naechst:.2f}: Val Acc={y_naechst:.4f} "
          f"[UCB={ucb.max():.4f}]")

bester_idx = np.argmax(y_beobachtet)
print(f"\nBeste Lernrate: 10^{X_beobachtet[bester_idx]:.3f} = {10**X_beobachtet[bester_idx]:.6f}")
print(f"Beste Val Accuracy: {y_beobachtet[bester_idx]*100:.2f}%")

# Visualisierung
gp.fit(X_beobachtet, y_beobachtet)
mu_pred, var_pred = gp.predict(suchraum.reshape(-1, 1))

plt.figure(figsize=(10, 5))
plt.fill_between(suchraum, mu_pred - 2*np.sqrt(var_pred),
                  mu_pred + 2*np.sqrt(var_pred), alpha=0.2, color='#00E6FF',
                  label='±2σ')
plt.plot(suchraum, mu_pred, color='#0082FF', linewidth=2, label='GP Vorhersage')
plt.scatter(X_beobachtet[:3], y_beobachtet[:3], c='green', s=100, zorder=5,
            label='Initial (3)')
plt.scatter(X_beobachtet[3:], y_beobachtet[3:], c='red', s=100, marker='*',
            zorder=5, s=200, label='BO Schritte')
plt.scatter(X_beobachtet[bester_idx], y_beobachtet[bester_idx],
            c='gold', s=300, marker='★', zorder=6, label='Bestes Ergebnis')
plt.xlabel('log10(Lernrate)')
plt.ylabel('Validation Accuracy')
plt.title('Bayesianische Optimierung: GP-Surrogate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('bayesianische_optimierung.png', dpi=150)
plt.show()


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 13: Hyperparameter-Optimierung
Niveau: Fortgeschrittene
Aufgabe 13.2: Keras Tuner – Automatische Architektursuche

Lernziel: Keras Tuner für automatisierte Hyperparameter-Suche
Datensatz: MNIST
"""

import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# Keras Tuner installieren falls nötig
try:
    import keras_tuner as kt
    print("Keras Tuner verfügbar!")
except ImportError:
    print("Installiere keras_tuner...")
    import subprocess
    subprocess.run(['pip', 'install', 'keras-tuner', '-q'])
    import keras_tuner as kt

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 784).astype('float32') / 255.0
X_sub = X_train[:8000]
y_sub = y_train[:8000]

def modell_ersteller(hp):
    """Hyperparameter-Raum definieren."""
    model = keras.Sequential()
    
    # Anzahl Schichten
    n_schichten = hp.Int('n_schichten', min_value=1, max_value=3, step=1)
    for i in range(n_schichten):
        einheiten = hp.Int(f'einheiten_{i}',
                           min_value=32, max_value=256, step=32)
        model.add(keras.layers.Dense(einheiten, activation='relu',
                                      input_shape=(784,) if i == 0 else []))
        
        if hp.Boolean(f'dropout_{i}'):
            rate = hp.Float(f'dropout_rate_{i}', min_value=0.1, max_value=0.5)
            model.add(keras.layers.Dropout(rate))
    
    model.add(keras.layers.Dense(10, activation='softmax'))
    
    # Optimizer
    lr = hp.Float('lernrate', min_value=1e-4, max_value=1e-1, sampling='log')
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Tuner konfigurieren
tuner = kt.RandomSearch(
    modell_ersteller,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='/tmp/keras_tuner',
    project_name='mnist_tuning',
    overwrite=True
)

tuner.search_space_summary()

print("\n=== Keras Tuner Suche ===")
stop_early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
tuner.search(X_sub, y_sub, epochs=15, validation_split=0.2, callbacks=[stop_early], verbose=0)

# Beste Hyperparameter
beste_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("\nBeste Hyperparameter:")
for hp_name, hp_val in beste_hps.values.items():
    print(f"  {hp_name}: {hp_val}")

# Bestes Modell trainieren
bestes_modell = tuner.hypermodel.build(beste_hps)
history = bestes_modell.fit(X_sub, y_sub, epochs=20, validation_split=0.2, verbose=1)

acc = bestes_modell.evaluate(X_test, y_test, verbose=0)[1]
print(f"\nTest Accuracy (bestes Modell): {acc*100:.2f}%")

# Top-3 Modelle vergleichen
plt.figure(figsize=(10, 4))
beste_modelle = tuner.get_best_hyperparameters(num_trials=3)
for i, hps in enumerate(beste_modelle):
    m = tuner.hypermodel.build(hps)
    h = m.fit(X_sub, y_sub, epochs=10, validation_split=0.2, verbose=0)
    plt.plot(h.history['val_accuracy'], label=f'Trial {i+1}')
plt.title('Top-3 Modelle (Keras Tuner)')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('keras_tuner.png', dpi=150)
plt.show()


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 13: Hyperparameter-Optimierung
Niveau: Fortgeschrittene
Aufgabe 13.3: Hyperparameter Sensitivity Analysis

Lernziel: Welche Hyperparameter sind am wichtigsten?
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

(X_train, y_train), _ = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_sub = X_train[:3000]
y_sub = y_train[:3000]

np.random.seed(42)

def evaluiere_konfiguration(hp_dict):
    model = keras.Sequential()
    model.add(keras.layers.Dense(hp_dict['einheiten_1'], activation='relu', input_shape=(784,)))
    if hp_dict['dropout']:
        model.add(keras.layers.Dropout(hp_dict['dropout_rate']))
    model.add(keras.layers.Dense(hp_dict['einheiten_2'], activation='relu'))
    if hp_dict['batch_norm']:
        model.add(keras.layers.BatchNormalization())
    model.add(keras.layers.Dense(10, activation='softmax'))
    
    model.compile(
        optimizer=keras.optimizers.Adam(hp_dict['lernrate']),
        loss='sparse_categorical_crossentropy', metrics=['accuracy']
    )
    h = model.fit(X_sub, y_sub, epochs=10, batch_size=int(hp_dict['batch_size']),
                  validation_split=0.2, verbose=0)
    return max(h.history['val_accuracy'])

# Zufällige HP-Konfigurationen abtasten
N = 30
print(f"Abtasten von {N} Konfigurationen...")
konfigurationen = []
for _ in range(N):
    hp = {
        'lernrate':   10 ** np.random.uniform(-4, -1),
        'einheiten_1': np.random.choice([32, 64, 128, 256]),
        'einheiten_2': np.random.choice([16, 32, 64, 128]),
        'dropout':     np.random.choice([0, 1]),
        'dropout_rate': np.random.uniform(0.1, 0.5),
        'batch_norm':  np.random.choice([0, 1]),
        'batch_size':  np.random.choice([32, 64, 128]),
    }
    acc = evaluiere_konfiguration(hp)
    konfigurationen.append({**hp, 'val_acc': acc})
    print(f"  {len(konfigurationen)}/{N}: {acc:.4f}")

# Als Tabelle
X_hp = np.array([[k['lernrate'], k['einheiten_1'], k['einheiten_2'],
                   k['dropout'], k['dropout_rate'], k['batch_norm'], k['batch_size']]
                  for k in konfigurationen])
y_hp = np.array([k['val_acc'] for k in konfigurationen])

hp_namen = ['lernrate', 'einheiten_1', 'einheiten_2', 'dropout',
            'dropout_rate', 'batch_norm', 'batch_size']

# Random Forest als Surrogat-Modell
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_hp, y_hp)
importances = rf.feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature Importance
sorted_idx = np.argsort(importances)[::-1]
axes[0].bar(range(len(hp_namen)),
             importances[sorted_idx],
             color='#00E6FF')
axes[0].set_xticks(range(len(hp_namen)))
axes[0].set_xticklabels([hp_namen[i] for i in sorted_idx], rotation=45, ha='right')
axes[0].set_title('Hyperparameter Wichtigkeit (RF Feature Importance)')
axes[0].set_ylabel('Wichtigkeit')
axes[0].grid(True, alpha=0.3, axis='y')

# Korrelation LR vs. Val Acc
axes[1].scatter(np.log10([k['lernrate'] for k in konfigurationen]),
                 y_hp, alpha=0.7, color='#7832FF', s=50)
axes[1].set_xlabel('log10(Lernrate)')
axes[1].set_ylabel('Validation Accuracy')
axes[1].set_title('Lernrate vs. Accuracy (wichtigster HP)')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Hyperparameter Sensitivity Analysis')
plt.tight_layout()
plt.savefig('hp_sensitivity.png', dpi=150)
plt.show()

print("\nWichtigste Hyperparameter:")
for i in sorted_idx[:3]:
    print(f"  {hp_namen[i]}: {importances[i]:.4f}")
